In [1]:
# =========================================
# Import packages and set up Neo4j
# =========================================
from dotenv import load_dotenv
import os
import textwrap

# LangChain (modern)
from langchain_community.graphs import Neo4jGraph
from langchain_openai import ChatOpenAI

# LCEL
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Warning control
import warnings
warnings.filterwarnings("ignore")

# Load env
load_dotenv('.env', override=True)
NEO4J_URI = os.getenv('NEO4J_URI')
NEO4J_USERNAME = os.getenv('NEO4J_USERNAME')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')
NEO4J_DATABASE = os.getenv('NEO4J_DATABASE') or 'neo4j'

# Models
llm = ChatOpenAI(temperature=0)

# Neo4j
kg = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database=NEO4J_DATABASE
)

In [2]:
# =========================================
# Explore the updated SEC documents graph
# =========================================
kg.refresh_schema()
print(textwrap.fill(kg.schema, 60))

kg.query("""
MATCH (mgr:Manager)-[:LOCATED_AT]->(addr:Address)
RETURN mgr, addr LIMIT 1
""")

kg.query("""
CALL db.index.fulltext.queryNodes("fullTextManagerNames", "royal bank")
YIELD node, score
RETURN node.managerName, score LIMIT 1
""")

Node properties: Chunk {textEmbedding: LIST, f10kItem:
STRING, chunkSeqId: INTEGER, text: STRING, cik: STRING,
cusip6: STRING, names: LIST, formId: STRING, source: STRING,
chunkId: STRING} Form {cusip6: STRING, names: LIST, formId:
STRING, source: STRING} Company {location: POINT, cusip:
STRING, names: LIST, companyAddress: STRING, companyName:
STRING, cusip6: STRING} Manager {location: POINT,
managerName: STRING, managerCik: STRING, managerAddress:
STRING} Address {location: POINT, country: STRING, city:
STRING, state: STRING} Relationship properties: SECTION
{f10kItem: STRING} OWNS_STOCK_IN {shares: INTEGER,
reportCalendarOrQuarter: STRING, value: FLOAT} The
relationships: (:Chunk)-[:NEXT]->(:Chunk)
(:Chunk)-[:PART_OF]->(:Form) (:Form)-[:SECTION]->(:Chunk)
(:Company)-[:FILED]->(:Form)
(:Company)-[:LOCATED_AT]->(:Address)
(:Manager)-[:LOCATED_AT]->(:Address)
(:Manager)-[:OWNS_STOCK_IN]->(:Company)


[{'node.managerName': 'Royal Bank of Canada', 'score': 4.431276321411133}]

In [3]:
# =========================================
# Cypher exploration queries (unchanged)
# =========================================
kg.query("""
MATCH (mgr:Manager)-[:LOCATED_AT]->(address:Address)
RETURN address.state, count(*) ORDER BY count(*) DESC LIMIT 10
""")

kg.query("""
MATCH (com:Company)-[:LOCATED_AT]->(address:Address)
RETURN address.state, count(*) ORDER BY count(*) DESC
""")

[{'address.state': 'California', 'count(*)': 7},
 {'address.state': 'Delaware', 'count(*)': 1},
 {'address.state': 'New York', 'count(*)': 1},
 {'address.state': 'Oregon', 'count(*)': 1}]

In [4]:
# =========================================
# 🔥 Modern Cypher Generation (LCEL)
# =========================================

cypher_prompt = ChatPromptTemplate.from_template("""
You are an expert Neo4j Cypher generator.

Use ONLY the schema below.

Schema:
{schema}

Rules:
- Only output Cypher
- No explanations
- No extra text

Examples:

# Investment firms in San Francisco
MATCH (mgr:Manager)-[:LOCATED_AT]->(a:Address)
WHERE a.city = "San Francisco"
RETURN mgr.managerName

# Firms near Santa Clara
MATCH (a:Address)
WHERE a.city = "Santa Clara"
MATCH (mgr:Manager)-[:LOCATED_AT]->(ma:Address)
WHERE point.distance(a.location, ma.location) < 10000
RETURN mgr.managerName

Question:
{question}
""")

cypher_chain = (
    {
        "schema": lambda _: kg.schema,
        "question": lambda x: x
    }
    | cypher_prompt
    | llm
    | StrOutputParser()
)


In [5]:
# =========================================
# Execute Cypher + Return Answer
# =========================================
def ask_graph(question: str):
    cypher = cypher_chain.invoke(question)
    print("\nGenerated Cypher:\n", cypher)

    try:
        result = kg.query(cypher)
        print("\nResult:")
        for r in result[:5]:
            print(r)
    except Exception as e:
        print("Error executing query:", e)

In [6]:
# =========================================
# Test Queries
# =========================================
ask_graph("What investment firms are in San Francisco?")
ask_graph("What companies are in Santa Clara?")
ask_graph("What investment firms are near Santa Clara?")


Generated Cypher:
 MATCH (mgr:Manager)-[:LOCATED_AT]->(a:Address)
WHERE a.city = "San Francisco"
RETURN mgr.managerName

Result:
{'mgr.managerName': 'PARNASSUS INVESTMENTS, LLC'}
{'mgr.managerName': 'SKBA CAPITAL MANAGEMENT LLC'}
{'mgr.managerName': 'ROSENBLUM SILVERMAN SUTTON S F INC /CA'}
{'mgr.managerName': 'CHARLES SCHWAB INVESTMENT MANAGEMENT INC'}
{'mgr.managerName': 'WELLS FARGO & COMPANY/MN'}

Generated Cypher:
 MATCH (c:Company)-[:LOCATED_AT]->(a:Address)
WHERE a.city = "Santa Clara"
RETURN c.companyName

Result:
{'c.companyName': 'PALO ALTO NETWORKS INC'}
{'c.companyName': 'SEAGATE TECHNOLOGY'}
{'c.companyName': 'ATLASSIAN CORP PLC'}

Generated Cypher:
 MATCH (a:Address)
WHERE a.city = "Santa Clara"
MATCH (mgr:Manager)-[:LOCATED_AT]->(ma:Address)
WHERE point.distance(a.location, ma.location) < 10000
RETURN mgr.managerName

Result:
{'mgr.managerName': 'Mine & Arao Wealth Creation & Management, LLC.'}


In [18]:
# =========================================
# Expand Cypher with Form 10K logic
# =========================================

cypher_prompt = ChatPromptTemplate.from_template("""
You are an expert Neo4j Cypher generator.

STRICT RULES:
- Output ONLY valid Cypher
- No explanations
- No extra text
- NEVER match Company using companyName directly
- ALWAYS use fulltext index for company lookup

MANDATORY PATTERN FOR COMPANIES:

CALL db.index.fulltext.queryNodes("fullTextCompanyNames", "<company>")
YIELD node
WITH node AS com

Then continue graph traversal.

Schema:
{schema}

Examples:

# What does Palo Alto Networks do?
CALL db.index.fulltext.queryNodes("fullTextCompanyNames", "Palo Alto Networks")
YIELD node
WITH node AS com
MATCH (com)-[:FILED]->(f:Form),
      (f)-[:SECTION]->(c:Chunk)
WHERE c.f10kItem = "item1"
RETURN c.text

Question:
{question}
""")

cypher_chain = (
    {
        "schema": lambda _: kg.schema,
        "question": lambda x: x
    }
    | cypher_prompt
    | llm
    | StrOutputParser()
)


ask_graph("What does Palo Alto Networks do?")





Generated Cypher:
 CALL db.index.fulltext.queryNodes("fullTextCompanyNames", "Palo Alto Networks")
YIELD node
WITH node AS com
MATCH (com)-[:FILED]->(f:Form),
      (f)-[:SECTION]->(c:Chunk)
WHERE c.f10kItem = "item1"
RETURN c.text

Result:
{'c.text': '>Item 1. Business\nGeneral\nPalo Alto Networks, Inc. is a global cybersecurity provider with a vision of a world where each day is safer and more secure than the one before. We were incorporated in 2005 and are headquartered in Santa Clara, California.\nWe empower enterprises, organizations, service providers, and government entities to protect themselves against today’s most sophisticated cyber threats. Our cybersecurity platforms and services help secure enterprise users, networks, clouds, and endpoints by delivering comprehensive cybersecurity backed by industry-leading artificial intelligence and automation. We are a leading provider of zero trust solutions, starting with next-generation zero trust network access to secure today’s r

In [21]:
# =========================================
# Answer Prompt (STRICT + RELIABLE)
# =========================================
answer_prompt = ChatPromptTemplate.from_template("""
You are answering questions based ONLY on the provided data.

STRICT RULES:
- If data is empty → say "No data found"
- If data contains results → use them
- Do NOT hallucinate
- Be concise and clear

Data:
{data}

Question:
{question}
""")

answer_chain = (
    answer_prompt
    | llm
    | StrOutputParser()
)

In [22]:
# =========================================
# 🔀 Query Router (CORE LOGIC)
# =========================================
def route_question(question: str) -> str:
    q = question.lower()

    if "what does" in q or "tell me about" in q:
        return "company"

    if "investment firms" in q or "firms" in q or "managers" in q:
        return "manager"

    return "default"

In [23]:
# =========================================
# 🏢 Company Query (FULLTEXT - SAFE)
# =========================================
def company_query(company_name: str) -> str:
    return f"""
    CALL db.index.fulltext.queryNodes("fullTextCompanyNames", "{company_name}")
    YIELD node
    WITH node AS com
    MATCH (com)-[:FILED]->(f:Form),
          (f)-[:SECTION]->(c:Chunk)
    WHERE c.f10kItem = "item1"
    RETURN c.text
    """

In [24]:
# =========================================
# 🧑‍💼 Manager Query (STRUCTURED)
# =========================================
def manager_query(city: str) -> str:
    return f"""
    MATCH (mgr:Manager)-[:LOCATED_AT]->(a:Address)
    WHERE a.city = "{city}"
    RETURN mgr.managerName
    """

In [25]:
# =========================================
# 🧠 Cypher Generator (ROUTED)
# =========================================
def generate_cypher(question: str) -> str:
    route = route_question(question)
    q = question.lower()

    # Company-based questions
    if route == "company":
        if "palo alto networks" in q:
            return company_query("Palo Alto Networks")

    # Manager/location-based questions
    if route == "manager":
        if "san francisco" in q:
            return manager_query("San Francisco")

    # Fallback (optional: you can plug LLM here later)
    return None

In [26]:
# =========================================
# 💬 Main Chat Function
# =========================================
def ask_graph_nl(question: str):
    cypher = generate_cypher(question)

    if not cypher:
        print("⚠️ No matching query pattern found.")
        return

    print("\nGenerated Cypher:\n", cypher)

    try:
        result = kg.query(cypher)
    except Exception as e:
        print("⚠️ Execution Error:", e)
        return

    # Debug visibility (VERY useful)
    print("\nRaw Result Sample:", result[:2])

    # Generate natural language answer
    response = answer_chain.invoke({
        "data": str(result),
        "question": question
    })

    print("\nAnswer:")
    print(textwrap.fill(response, 80))

In [27]:
# =========================================
# 🧪 Test Queries
# =========================================
ask_graph_nl("What investment firms are in San Francisco?")
ask_graph_nl("What does Palo Alto Networks do?")


Generated Cypher:
 
    MATCH (mgr:Manager)-[:LOCATED_AT]->(a:Address)
    WHERE a.city = "San Francisco"
    RETURN mgr.managerName
    

Raw Result Sample: [{'mgr.managerName': 'PARNASSUS INVESTMENTS, LLC'}, {'mgr.managerName': 'SKBA CAPITAL MANAGEMENT LLC'}]

Answer:
The investment firms in San Francisco are: - ROSENBLUM SILVERMAN SUTTON S F INC
/CA - Robertson Stephens Wealth Management, LLC - Pacific Heights Asset
Management LLC - Jacobs & Co/CA

Generated Cypher:
 
    CALL db.index.fulltext.queryNodes("fullTextCompanyNames", "Palo Alto Networks")
    YIELD node
    WITH node AS com
    MATCH (com)-[:FILED]->(f:Form),
          (f)-[:SECTION]->(c:Chunk)
    WHERE c.f10kItem = "item1"
    RETURN c.text
    

Raw Result Sample: [{'c.text': '>Item 1. Business\nGeneral\nPalo Alto Networks, Inc. is a global cybersecurity provider with a vision of a world where each day is safer and more secure than the one before. We were incorporated in 2005 and are headquartered in Santa Clara, Cal